In [1]:
import torch
import torchvision
from torchvision import datasets, transforms

# Baseline model

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms

import torch.nn as nn
import torch.nn.functional as F

transform = transforms.Compose(
    [transforms.Resize([300,300]), # resize the image to its stock size of 300x300
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# we have 4500 training images, thus 45 batch size gives us 100 batches
batch_size = 45

# read in and categorize data in given directory
trainset = datasets.ImageFolder('C:/traindata1/traindata', transform=transform) 
testset = datasets.ImageFolder('C:/traindata1/testdata', transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False)
# create a trainloader with shuffle = True so as to make sure our training does not generalize to one class too much
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True)
# create our classification class 'Net'
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5) # take our 3 channel data and change to 6 channel with a kernal size of 5
        self.pool = nn.MaxPool2d(2, 2) # giving a pooling size of 2,2 means that our image size is halved
        self.conv2 = nn.Conv2d(6, 16, 5) # take our 6 channel data and change it to 16 channel data with same kernal size 5
        # based on kernal size, stride, padding and image size we get the below calculation
        self.fc1 = nn.Linear(16 * 72 * 72, 120) 
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3) # must have 3 here as we want results for 3 classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) 
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net() # create an instance of our classifier

import torch.optim as optim

criterion = nn.CrossEntropyLoss() # create our loss funnction
optimizer = optim.Rprop(net.parameters(), lr=0.09) # create our optimizer with our classifier parameters

for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')



correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 1500 test images: {100 * correct // total} %')

Finished Training
Accuracy of the network on the 1500 test images: 34 %


# Actual model

In [3]:
classes = ['cherry', 'strawberry', 'tomato']

In [9]:
import torch.nn as nn
import torch.nn.functional as F

transform = transforms.Compose(
    [transforms.Resize([300,300]), # resize the image to its stock size of 300x300
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# we have 4500 training images, thus 45 batch size gives us 100 batches
batch_size = 45

# read in and categorize data in given directory
trainset = datasets.ImageFolder('C:/traindata1/traindata2', transform=transform) 

# create a trainloader with shuffle = True so as to make sure our training does not generalize to one class too much
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True)
# create our classification class 'Net'
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5) # take our 3 channel data and change to 6 channel with a kernal size of 5
        self.pool = nn.MaxPool2d(2, 2) # giving a pooling size of 2,2 means that our image size is halved
        self.conv2 = nn.Conv2d(6, 16, 5) # take our 6 channel data and change it to 16 channel data with same kernal size 5
        # based on kernal size, stride, padding and image size we get the below calculation
        self.fc1 = nn.Linear(16 * 72 * 72, 120) 
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3) # must have 3 here as we want results for 3 classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) 
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net() # create an instance of our classifier

In [10]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss() # create our loss funnction
optimizer = optim.Adamax(net.parameters(), lr=0.008) # create our optimizer with our classifier parameters

In [11]:
for epoch in range(3):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')
#Adamax

Finished Training


In [12]:
# save our model
PATH = 'model.pth'
torch.save(net.state_dict(), PATH)